In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# CREATE TEMPERATURE INPUTS FOR SEASONAL IDEAL CASE STUDIES
# ============================================================
#
# Input folder structure:
#
# C:/IDEAL_Programming/New_Evaluation/SEASONAL/
#   summer_home100_20170809/
#       history_store.csv
#       actual_target_day.csv
#       case_metadata.json
#
# Output per case folder:
#   temperature_inputs.csv
#   temperature_inputs_for_ui.txt
#   temperature_inputs_metadata.json
#
# Main UI recommendation:
#   external_temperature_24h -> observed target-day external temperature
#   internal_temperature_24h -> previous 7-day hourly median internal temperature
#
# Observed target-day internal temperature is saved only as oracle/diagnostic.
#
# ============================================================

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = Path(r"C:/IDEAL_Programming/processed/final/IDEAL_final_hourly_dataset.csv")
SEASONAL_ROOT = Path(r"C:/IDEAL_Programming/New_Evaluation/SEASONAL")

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

HISTORY_DAYS = 7
EXPECTED_TARGET_HOURS = 24
EXPECTED_HISTORY_HOURS = 168

SAVE_ENCODING = "utf-8-sig"

EXTERNAL_TEMP_CANDIDATES = [
    "external_temperature",
    "external_temperature_C",
    "external_temp_C",
    "temperature",
    "temperature_C",
    "outdoor_temperature",
    "outdoor_temperature_C",
]

INTERNAL_TEMP_CANDIDATES = [
    "internal_temperature",
    "internal_temperature_C",
    "internal_temp_C",
    "inside_temperature",
    "inside_temperature_C",
    "indoor_temperature",
    "indoor_temperature_C",
]

STATIC_COL_CANDIDATES = [
    "hometype",
    "residents",
    "urban_rural_class",
    "total_floor_area_m2",
    "num_electric_appliances",
]


# ============================================================
# HELPERS
# ============================================================

def find_first_existing_col(df, candidates, required=True, label="column"):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(
            f"Could not find {label}. Tried: {candidates}\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def mode_or_nan(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]


def comma_string(values, decimals=2):
    out = []

    for value in values:
        if pd.isna(value):
            out.append("")
        else:
            out.append(f"{float(value):.{decimals}f}")

    return ", ".join(out)


def json_safe(value):
    if isinstance(value, pd.Timestamp):
        return str(value)

    if isinstance(value, (np.integer, np.int64)):
        return int(value)

    if isinstance(value, (np.floating, np.float64)):
        return None if pd.isna(value) else float(value)

    if isinstance(value, float) and pd.isna(value):
        return None

    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_safe(v) for v in value]

    return value


def prepare_home_hourly(df, value_cols):
    keep_cols = [HOME_COL, TIME_COL] + value_cols
    keep_cols = [c for c in keep_cols if c in df.columns]

    tmp = df[keep_cols].copy()
    tmp[TIME_COL] = pd.to_datetime(tmp[TIME_COL])

    for col in value_cols:
        if col in tmp.columns:
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    agg_dict = {
        col: "mean"
        for col in value_cols
        if col in tmp.columns
    }

    tmp = (
        tmp
        .groupby([HOME_COL, TIME_COL], as_index=False)
        .agg(agg_dict)
        .sort_values([HOME_COL, TIME_COL])
        .reset_index(drop=True)
    )

    return tmp


def make_24h_target_vector(home_df, value_col, target_start):
    target_end = target_start + pd.Timedelta(days=1)

    expected_index = pd.date_range(
        target_start,
        target_end - pd.Timedelta(hours=1),
        freq="h",
    )

    tmp = home_df[
        (home_df[TIME_COL] >= target_start) &
        (home_df[TIME_COL] < target_end)
    ][[TIME_COL, value_col]].copy()

    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")

    tmp = (
        tmp
        .groupby(TIME_COL, as_index=False)
        .agg({value_col: "mean"})
        .set_index(TIME_COL)
        .reindex(expected_index)
    )

    tmp.index.name = TIME_COL
    tmp = tmp.reset_index()
    tmp["hour"] = tmp[TIME_COL].dt.hour

    missing_before_fill = int(tmp[value_col].isna().sum())

    tmp[value_col] = (
        tmp[value_col]
        .interpolate(method="linear", limit_direction="both")
        .ffill()
        .bfill()
    )

    missing_after_fill = int(tmp[value_col].isna().sum())

    return tmp, missing_before_fill, missing_after_fill


def make_7d_hourly_median_profile(home_df, value_col, history_start, history_end):
    hist = home_df[
        (home_df[TIME_COL] >= history_start) &
        (home_df[TIME_COL] < history_end)
    ][[TIME_COL, value_col]].copy()

    hist[value_col] = pd.to_numeric(hist[value_col], errors="coerce")
    hist["hour"] = hist[TIME_COL].dt.hour

    profile = (
        hist
        .groupby("hour", as_index=False)
        .agg(
            median_7d=(value_col, "median"),
            mean_7d=(value_col, "mean"),
            std_7d=(value_col, "std"),
            observed_count=(value_col, "count"),
        )
        .set_index("hour")
        .reindex(range(24))
        .reset_index()
    )

    missing_before_fill = int(profile["median_7d"].isna().sum())

    profile["median_7d"] = (
        profile["median_7d"]
        .interpolate(method="linear", limit_direction="both")
        .ffill()
        .bfill()
    )

    missing_after_fill = int(profile["median_7d"].isna().sum())

    return profile, missing_before_fill, missing_after_fill


def extract_static_metadata(df, home_id):
    available_static_cols = [c for c in STATIC_COL_CANDIDATES if c in df.columns]
    hdf = df[df[HOME_COL] == home_id].copy()

    row = {"home_id": home_id}

    for col in available_static_cols:
        if pd.api.types.is_numeric_dtype(hdf[col]):
            row[col] = float(pd.to_numeric(hdf[col], errors="coerce").median())
        else:
            row[col] = mode_or_nan(hdf[col])

    return row


def load_case_metadata(case_dir):
    meta_path = case_dir / "case_metadata.json"

    if not meta_path.exists():
        return None

    with open(meta_path, "r", encoding="utf-8") as f:
        return json.load(f)


def find_case_dirs(root):
    case_dirs = []

    for p in root.iterdir():
        if not p.is_dir():
            continue

        if (p / "case_metadata.json").exists():
            case_dirs.append(p)

    return sorted(case_dirs)


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 120)
print("Loading IDEAL final hourly dataset")
print("=" * 120)

raw_df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)

external_col = find_first_existing_col(
    raw_df,
    EXTERNAL_TEMP_CANDIDATES,
    required=True,
    label="external temperature column",
)

internal_col = find_first_existing_col(
    raw_df,
    INTERNAL_TEMP_CANDIDATES,
    required=False,
    label="internal temperature column",
)

value_cols = [external_col]
if internal_col is not None:
    value_cols.append(internal_col)

hourly_df = prepare_home_hourly(raw_df, value_cols=value_cols)

print("Rows:", len(raw_df))
print("Homes:", raw_df[HOME_COL].nunique())
print("Date range:", raw_df[TIME_COL].min(), "to", raw_df[TIME_COL].max())
print("External temperature column:", external_col)
print("Internal temperature column:", internal_col)
print("Seasonal root:", SEASONAL_ROOT)

case_dirs = find_case_dirs(SEASONAL_ROOT)

if not case_dirs:
    raise ValueError(f"No seasonal case folders found in: {SEASONAL_ROOT}")

print("Cases found:")
for p in case_dirs:
    print(" -", p.name)

summary_rows = []


# ============================================================
# PROCESS CASES
# ============================================================

for case_dir in case_dirs:
    metadata = load_case_metadata(case_dir)

    if metadata is None:
        print(f"[WARN] Missing metadata in {case_dir}")
        continue

    season = metadata.get("season", case_dir.name.split("_", 1)[0])
    home_id = metadata["home_id"]
    target_start = pd.Timestamp(metadata["target_date"]).normalize()
    target_end = target_start + pd.Timedelta(days=1)

    history_start = target_start - pd.Timedelta(days=HISTORY_DAYS)
    history_end = target_start

    day_name_gr = metadata.get("day_name_gr")
    is_weekend = metadata.get("is_weekend")

    home_df = hourly_df[hourly_df[HOME_COL] == home_id].copy()

    if home_df.empty:
        print(f"[WARN] {case_dir.name}: no rows found for {home_id}.")
        continue

    # --------------------------------------------------------
    # External target-day observed 24h values
    # --------------------------------------------------------
    external_24, ext_missing_before, ext_missing_after = make_24h_target_vector(
        home_df=home_df,
        value_col=external_col,
        target_start=target_start,
    )

    external_24 = external_24.rename(
        columns={external_col: "external_temperature_24h_observed"}
    )

    out = external_24[
        [TIME_COL, "hour", "external_temperature_24h_observed"]
    ].copy()

    # --------------------------------------------------------
    # Internal target-day observed 24h values
    # Diagnostic/oracle only
    # --------------------------------------------------------
    int_oracle_missing_before = None
    int_oracle_missing_after = None
    internal_oracle_text = ""

    if internal_col is not None:
        internal_target_24, int_oracle_missing_before, int_oracle_missing_after = make_24h_target_vector(
            home_df=home_df,
            value_col=internal_col,
            target_start=target_start,
        )

        internal_target_24 = internal_target_24.rename(
            columns={internal_col: "internal_temperature_24h_observed_oracle"}
        )

        out = out.merge(
            internal_target_24[
                [TIME_COL, "internal_temperature_24h_observed_oracle"]
            ],
            on=TIME_COL,
            how="left",
        )

    # --------------------------------------------------------
    # Recommended internal 24h values:
    # previous 7-day median by hour
    # --------------------------------------------------------
    internal_recommended_text = ""
    internal_recommended_missing_before = None
    internal_recommended_missing_after = None

    if internal_col is not None:
        internal_profile, internal_recommended_missing_before, internal_recommended_missing_after = make_7d_hourly_median_profile(
            home_df=home_df,
            value_col=internal_col,
            history_start=history_start,
            history_end=history_end,
        )

        internal_profile = internal_profile.rename(
            columns={
                "median_7d": "internal_temperature_24h_7d_median_recommended",
                "mean_7d": "internal_temperature_7d_mean",
                "std_7d": "internal_temperature_7d_std",
                "observed_count": "internal_temperature_7d_observed_count",
            }
        )

        out = out.merge(
            internal_profile[
                [
                    "hour",
                    "internal_temperature_24h_7d_median_recommended",
                    "internal_temperature_7d_mean",
                    "internal_temperature_7d_std",
                    "internal_temperature_7d_observed_count",
                ]
            ],
            on="hour",
            how="left",
        )

    # --------------------------------------------------------
    # Add static/case metadata
    # --------------------------------------------------------
    static_meta = extract_static_metadata(raw_df, home_id)

    out["season"] = season
    out["case_folder"] = case_dir.name
    out["home_id"] = home_id
    out["target_date"] = target_start.date().isoformat()
    out["day_name_gr"] = day_name_gr
    out["is_weekend"] = is_weekend

    out["history_stability_score"] = metadata.get("history_stability_score")
    out["history_stability_category"] = metadata.get("history_stability_category")
    out["recommended_max_alpha"] = metadata.get("recommended_max_alpha")
    out["recommended_max_alpha_range"] = metadata.get("recommended_max_alpha_range")

    for key, value in static_meta.items():
        if key != HOME_COL:
            out[key] = value

    out["history_start"] = history_start
    out["history_end_exclusive"] = history_end

    # --------------------------------------------------------
    # Reorder columns
    # --------------------------------------------------------
    preferred_cols = [
        "season",
        "case_folder",
        "home_id",
        "target_date",
        "day_name_gr",
        "is_weekend",
        TIME_COL,
        "hour",
        "external_temperature_24h_observed",
        "internal_temperature_24h_7d_median_recommended",
        "internal_temperature_24h_observed_oracle",
        "internal_temperature_7d_mean",
        "internal_temperature_7d_std",
        "internal_temperature_7d_observed_count",
        "history_stability_score",
        "history_stability_category",
        "recommended_max_alpha",
        "recommended_max_alpha_range",
        "hometype",
        "residents",
        "urban_rural_class",
        "total_floor_area_m2",
        "num_electric_appliances",
        "history_start",
        "history_end_exclusive",
    ]

    preferred_cols = [c for c in preferred_cols if c in out.columns]
    remaining_cols = [c for c in out.columns if c not in preferred_cols]
    out = out[preferred_cols + remaining_cols]

    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------
    temperature_csv = case_dir / "temperature_inputs.csv"
    out.to_csv(temperature_csv, index=False, encoding=SAVE_ENCODING)

    # UI-ready text
    external_text = comma_string(
        out["external_temperature_24h_observed"].tolist(),
        decimals=2,
    )

    if "internal_temperature_24h_7d_median_recommended" in out.columns:
        internal_recommended_text = comma_string(
            out["internal_temperature_24h_7d_median_recommended"].tolist(),
            decimals=2,
        )

    if "internal_temperature_24h_observed_oracle" in out.columns:
        internal_oracle_text = comma_string(
            out["internal_temperature_24h_observed_oracle"].tolist(),
            decimals=2,
        )

    ui_txt = case_dir / "temperature_inputs_for_ui.txt"

    with open(ui_txt, "w", encoding="utf-8") as f:
        f.write(f"season: {season}\n")
        f.write(f"case_folder: {case_dir.name}\n")
        f.write(f"home_id: {home_id}\n")
        f.write(f"target_date: {target_start.date().isoformat()}\n")
        f.write(f"day: {day_name_gr}\n")
        f.write(f"is_weekend: {is_weekend}\n")
        f.write(f"history_stability_score: {metadata.get('history_stability_score')}\n")
        f.write(f"history_stability_category: {metadata.get('history_stability_category')}\n")
        f.write(f"recommended_max_alpha: {metadata.get('recommended_max_alpha')}\n")
        f.write(f"recommended_max_alpha_range: {metadata.get('recommended_max_alpha_range')}\n")
        f.write("\n")

        f.write("Paste into external_temperature_24h:\n")
        f.write(external_text)
        f.write("\n\n")

        f.write("Recommended paste into internal_temperature_24h ")
        f.write("(previous 7-day hourly median; no target-day leakage):\n")
        f.write(internal_recommended_text if internal_recommended_text else "N/A")
        f.write("\n\n")

        f.write("Observed target-day internal_temperature_24h ")
        f.write("(ORACLE / diagnostic only; not recommended for main evaluation):\n")
        f.write(internal_oracle_text if internal_oracle_text else "N/A")
        f.write("\n")

    # JSON metadata
    temperature_metadata = {
        "season": season,
        "case_folder": case_dir.name,
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,

        "history_days": HISTORY_DAYS,
        "history_start": str(history_start),
        "history_end_exclusive": str(history_end),

        "history_stability_score": metadata.get("history_stability_score"),
        "history_stability_category": metadata.get("history_stability_category"),
        "recommended_max_alpha": metadata.get("recommended_max_alpha"),
        "recommended_max_alpha_range": metadata.get("recommended_max_alpha_range"),

        "external_temperature_column": external_col,
        "internal_temperature_column": internal_col,

        "external_missing_before_fill": int(ext_missing_before),
        "external_missing_after_fill": int(ext_missing_after),

        "internal_oracle_missing_before_fill": int_oracle_missing_before,
        "internal_oracle_missing_after_fill": int_oracle_missing_after,

        "internal_recommended_missing_before_fill": internal_recommended_missing_before,
        "internal_recommended_missing_after_fill": internal_recommended_missing_after,

        "temperature_csv": str(temperature_csv),
        "temperature_ui_text_file": str(ui_txt),

        "external_temperature_24h_for_ui": external_text,
        "internal_temperature_24h_recommended_for_ui": internal_recommended_text,
        "internal_temperature_24h_observed_oracle": internal_oracle_text,

        "static_metadata": static_meta,

        "methodological_note": (
            "External temperature uses observed target-day environmental data as case-study weather proxy. "
            "Internal temperature recommended input is derived from the previous 7-day hourly median "
            "to avoid target-day oracle leakage."
        ),
    }

    metadata_json = case_dir / "temperature_inputs_metadata.json"

    with open(metadata_json, "w", encoding="utf-8") as f:
        json.dump(json_safe(temperature_metadata), f, ensure_ascii=False, indent=2)

    summary_rows.append({
        "season": season,
        "case_folder": case_dir.name,
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,

        "hometype": static_meta.get("hometype", np.nan),
        "residents": static_meta.get("residents", np.nan),
        "urban_rural_class": static_meta.get("urban_rural_class", np.nan),
        "total_floor_area_m2": static_meta.get("total_floor_area_m2", np.nan),
        "num_electric_appliances": static_meta.get("num_electric_appliances", np.nan),

        "history_stability_score": metadata.get("history_stability_score"),
        "history_stability_category": metadata.get("history_stability_category"),
        "recommended_max_alpha": metadata.get("recommended_max_alpha"),
        "recommended_max_alpha_range": metadata.get("recommended_max_alpha_range"),

        "external_col": external_col,
        "internal_col": internal_col,
        "external_missing_before_fill": int(ext_missing_before),
        "external_missing_after_fill": int(ext_missing_after),
        "internal_recommended_missing_before_fill": internal_recommended_missing_before,
        "internal_recommended_missing_after_fill": internal_recommended_missing_after,

        "temperature_csv": str(temperature_csv),
        "temperature_ui_text_file": str(ui_txt),
        "metadata_json": str(metadata_json),

        "external_temperature_24h_for_ui": external_text,
        "internal_temperature_24h_recommended_for_ui": internal_recommended_text,
        "internal_temperature_24h_observed_oracle": internal_oracle_text,
    })

    print("=" * 120)
    print(f"{season.upper()} | {home_id} | {target_start.date()} | {day_name_gr}")
    print("=" * 120)
    print("Case folder:", case_dir.name)
    print(
        f"Static: hometype={static_meta.get('hometype', 'N/A')}, "
        f"residents={static_meta.get('residents', 'N/A')}, "
        f"urban_rural_class={static_meta.get('urban_rural_class', 'N/A')}, "
        f"floor_area={static_meta.get('total_floor_area_m2', 'N/A')}, "
        f"appliances={static_meta.get('num_electric_appliances', 'N/A')}"
    )
    print(
        f"Stability: {metadata.get('history_stability_category')} "
        f"({metadata.get('history_stability_score')}) | "
        f"Recommended alpha: {metadata.get('recommended_max_alpha')}"
    )
    print("External missing before/after fill:", ext_missing_before, "/", ext_missing_after)
    print("Internal recommended missing before/after fill:", internal_recommended_missing_before, "/", internal_recommended_missing_after)
    print("Saved CSV:", temperature_csv)
    print("Saved UI text:", ui_txt)
    print()


# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary = pd.DataFrame(summary_rows)

summary_file = SEASONAL_ROOT / "seasonal_temperature_inputs_summary.csv"
summary.to_csv(summary_file, index=False, encoding=SAVE_ENCODING)

print("=" * 120)
print("SEASONAL TEMPERATURE INPUTS SUMMARY")
print("=" * 120)

print_cols = [
    "season",
    "case_folder",
    "home_id",
    "target_date",
    "day_name_gr",
    "hometype",
    "residents",
    "total_floor_area_m2",
    "num_electric_appliances",
    "history_stability_score",
    "history_stability_category",
    "recommended_max_alpha",
    "external_missing_before_fill",
    "external_missing_after_fill",
    "internal_recommended_missing_before_fill",
    "internal_recommended_missing_after_fill",
    "temperature_ui_text_file",
]

print_cols = [c for c in print_cols if c in summary.columns]

print(summary[print_cols].to_string(index=False))

print("=" * 120)
print("Saved summary:")
print(summary_file)

Loading IDEAL final hourly dataset
Rows: 1529350
Homes: 254
Date range: 2016-08-10 10:00:00 to 2018-06-30 23:00:00
External temperature column: external_temperature
Internal temperature column: internal_temperature
Seasonal root: C:\IDEAL_Programming\New_Evaluation\SEASONAL
Cases found:
 - autumn_home72_20170927
 - spring_home316_20180423
 - summer_home100_20170809
 - winter_home268_20180111
AUTUMN | home72 | 2017-09-27 | Τετάρτη
Case folder: autumn_home72_20170927
Static: hometype=flat, residents=2.0, urban_rural_class=1, floor_area=80.0, appliances=9.0
Stability: Excellent (0.959708) | Recommended alpha: 0.6
External missing before/after fill: 0 / 0
Internal recommended missing before/after fill: 0 / 0
Saved CSV: C:\IDEAL_Programming\New_Evaluation\SEASONAL\autumn_home72_20170927\temperature_inputs.csv
Saved UI text: C:\IDEAL_Programming\New_Evaluation\SEASONAL\autumn_home72_20170927\temperature_inputs_for_ui.txt

SPRING | home316 | 2018-04-23 | Δευτέρα
Case folder: spring_home316_20